# D125 — Test-Driven Development

### A practical introduction in 30 minutes or less

TDD means writing a small test before writing the behavior.

## Learning goals

By the end, you can:

- compare top-down and bottom-up coding
- explain Red–Green–Refactor
- build one behavior through small TDD steps
- recognize useful and unhelpful TDD tests

## Two ways to begin coding

| Top-down | Bottom-up |
|---|---|
| begin with the whole workflow | begin with small building blocks |
| divide it into smaller parts | combine tested parts gradually |
| useful for design and planning | useful for implementation |

Real projects commonly use both.

## Top-down example

Start with the complete checkout workflow:

```text
checkout order
├── calculate item total
├── apply discount
├── calculate delivery fee
└── produce final amount
```

The overall goal is visible before every detail exists.

## Bottom-up example

Build and test small functions first:

```text
item_total() ──────┐
discount_amount() ─┼──> checkout()
delivery_fee() ────┘
```

Each building block works before it joins the workflow.

In [ ]:
def item_total(unit_price, quantity):
    return unit_price * quantity

assert item_total(500, 2) == 1000

## Where does TDD fit?

TDD supports small, bottom-up implementation steps:

1. Describe one behavior with a test.
2. Write only enough code to pass.
3. Improve the code while tests stay green.
4. Repeat for the next behavior.

## Red — Green — Refactor

| Stage | Action |
|---|---|
| 🔴 **Red** | write a test and see it fail correctly |
| 🟢 **Green** | write the smallest code that passes |
| 🔵 **Refactor** | improve structure without changing behavior |

Then start the loop again.

## Before Red: choose one requirement

Feature: calculate an order's delivery fee.

First requirement:

> Orders below ₹500 have a ₹50 delivery fee.

This is small enough for one test.

## 🔴 Red: write the first test

Write the expected behavior before the function exists.

```python
def test_delivery_below_500_costs_50():
    assert delivery_fee(300) == 50
```

It fails with `NameError`. That is the expected Red stage.

## 🟢 Green: smallest passing code

Implement only the behavior requested by the test.

In [ ]:
def delivery_fee(order_total):
    return 50

def test_delivery_below_500_costs_50():
    assert delivery_fee(300) == 50

test_delivery_below_500_costs_50()

## Why is that tiny code acceptable?

- It passes the current requirement.
- It avoids guessing future requirements.
- The next test will force the next improvement.

Green code may be incomplete. It must be correct for all tests written so far.

## 🔴 Red again: add a boundary rule

Second requirement:

> Orders of ₹500 or more receive free delivery.

```python
def test_delivery_is_free_at_500():
    assert delivery_fee(500) == 0
```

The current implementation returns `50`, so this test exposes missing behavior.

## 🟢 Green again

Add the smallest decision that passes both tests.

In [ ]:
def delivery_fee(order_total):
    if order_total >= 500:
        return 0
    return 50

assert delivery_fee(300) == 50
assert delivery_fee(500) == 0

## 🔵 Refactor

Improve clarity without changing behavior.

In [ ]:
FREE_DELIVERY_MINIMUM = 500
STANDARD_DELIVERY_FEE = 50

def delivery_fee(order_total):
    if order_total >= FREE_DELIVERY_MINIMUM:
        return 0
    return STANDARD_DELIVERY_FEE

assert delivery_fee(300) == 50
assert delivery_fee(500) == 0

## Add invalid-input behavior

Third requirement:

> A negative order total is invalid.

The next test expects a `ValueError`.

In [ ]:
def delivery_fee(order_total):
    if order_total < 0:
        raise ValueError("Order total cannot be negative")
    if order_total >= FREE_DELIVERY_MINIMUM:
        return 0
    return STANDARD_DELIVERY_FEE

In [ ]:
try:
    delivery_fee(-1)
    assert False, "Expected ValueError"
except ValueError as error:
    assert str(error) == "Order total cannot be negative"

## Add the next small building block

Requirement:

> Premium customers receive a 10% discount.

In [ ]:
def discount_amount(total, is_premium):
    return total * 0.10 if is_premium else 0

assert discount_amount(1000, True) == 100
assert discount_amount(1000, False) == 0

## Combine tested building blocks

Now the top-level checkout function can use the smaller functions.

In [ ]:
def checkout_total(item_total, is_premium=False):
    discount = discount_amount(item_total, is_premium)
    discounted_total = item_total - discount
    return discounted_total + delivery_fee(discounted_total)

assert checkout_total(1000, True) == 900
assert checkout_total(300, False) == 350

## Run all tests after every change

A new test checks new behavior.
Existing tests protect behavior already completed.

```text
new test + previous tests = growing safety net
```

In [ ]:
def run_delivery_tests():
    assert delivery_fee(0) == 50
    assert delivery_fee(499) == 50
    assert delivery_fee(500) == 0
    assert delivery_fee(1000) == 0

run_delivery_tests()

## TDD with unittest or pytest

TDD is a workflow, not a specific tool.

- plain assertions are enough to learn the cycle
- `unittest` is included with Python
- pytest provides convenient discovery and failure reports

Use the tool already selected by your project.

## Good first TDD scenarios

- total equals price × quantity
- free delivery begins at a boundary
- coupon reduces a total
- empty cart cannot checkout
- invalid quantity raises an exception

Choose behavior with clear input and output.

## Common mistakes

- writing many tests before running any
- skipping the Red stage
- adding more production code than the test requires
- refactoring while tests are failing
- testing implementation details instead of behavior
- making one test cover several unrelated rules

## A practical TDD rhythm

1. Select the smallest unfinished behavior.
2. Write one clear test.
3. Run it and confirm the expected failure.
4. Make it pass with minimal code.
5. Run all tests.
6. Refactor if clarity can improve.
7. Commit the small working change.

## Quick practice

Develop this rule using Red–Green–Refactor:

> `loyalty_points(total)` awards 1 point for every complete ₹100.

Suggested test order:

1. ₹50 gives 0 points
2. ₹100 gives 1 point
3. ₹250 gives 2 points
4. negative total raises `ValueError`

In [ ]:
def loyalty_points(total):
    if total < 0:
        raise ValueError("Total cannot be negative")
    return total // 100

assert loyalty_points(50) == 0
assert loyalty_points(100) == 1
assert loyalty_points(250) == 2

## Recap

- Top-down clarifies the whole workflow.
- Bottom-up builds small, tested parts.
- TDD follows Red–Green–Refactor.
- Make one small behavior pass at a time.
- Run all tests after every change.
- Refactor only when the tests are green.

## Present this notebook

Use VS Code's notebook slideshow support, or serve it with:

```powershell
jupyter nbconvert --to slides D125_TestDrivenDevelopment.ipynb --post serve
```